# Step 1: Install Dependencies

In [1]:
!pip install -q transformers accelerate bitsandbytes matplotlib seaborn
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared
!chmod +x cloudflared

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00:00:0100:01
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 37.2M  100 37.2M    0     0  31.3M      0  0:00:01  0:00:01 --:--:--  100M


# Step 2: Load Nayari

In [2]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = "Crossie/Nayari"

print("Loading model weights...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# Initialize pipeline for easier testing
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Loading model weights...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

# Step 3: Benchmark Tests

In [3]:
import time

def run_benchmark():
    tests = {
        "Logic": "If I have 3 apples and you take away 2, how many apples do you have?",
        "Coding": "Write a Python function to check if a number is prime.",
        "Creative": "Write a short haiku about a catgirl named Asmi.",
        "Reasoning": "Sally has 3 brothers. Each brother has 2 sisters. How many sisters does Sally have?"
    }
    
    results = {}
    
    print("🚀 Starting Benchmarks...")
    
    for category, prompt in tests.items():
        start_time = time.time()
        
        # Tokenize input
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        input_len = inputs['input_ids'].shape[1]
        
        # Generate
        output_tokens = model.generate(
            **inputs, 
            max_new_tokens=150, 
            temperature=0.7,
            do_sample=True
        )
        
        end_time = time.time()
        
        # Calculations
        duration = end_time - start_time
        new_tokens = output_tokens[0][input_len:]
        num_tokens = len(new_tokens)
        tps = num_tokens / duration
        
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        results[category] = {
            "tps": round(tps, 2),
            "response": response_text,
            "tokens": num_tokens
        }
        print(f"✅ {category} Complete ({round(tps, 2)} tokens/sec)")

    return results

# Run it
benchmark_data = run_benchmark()

🚀 Starting Benchmarks...
✅ Logic Complete (14.93 tokens/sec)
✅ Coding Complete (19.53 tokens/sec)
✅ Creative Complete (19.63 tokens/sec)
✅ Reasoning Complete (19.65 tokens/sec)


# Step 4: Export Graph as JPG

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

def save_benchmark_graph(results):
    categories = list(results.keys())
    tps_values = [results[c]['tps'] for c in categories]
    
    plt.figure(figsize=(12, 7))
    sns.set_style("whitegrid")
    
    # Create Bar Plot
    colors = sns.color_palette("magma", len(categories))
    bars = plt.bar(categories, tps_values, color=colors)
    
    plt.title("Nayari Model Performance Benchmark", fontsize=16)
    plt.ylabel("Tokens Per Second (TPS)", fontsize=12)
    plt.xlabel("Test Category", fontsize=12)
    
    # Add values on top of bars
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, yval, ha='center', va='bottom', fontweight='bold')
    
    # Save as JPG
    plt.savefig("benchmark_results.jpg", format='jpg', dpi=300, bbox_inches='tight')
    print("📊 Graph saved as 'benchmark_results.jpg'")
    plt.show()

save_benchmark_graph(benchmark_data)

NameError: name 'benchmark_data' is not defined

# Step 5: Output Text Logs

In [5]:
with open("model_outputs.txt", "w") as f:
    for cat, data in benchmark_data.items():
        f.write(f"=== {cat} (TPS: {data['tps']}) ===\n")
        f.write(f"Response: {data['response']}\n\n")
print("📝 Outputs saved to 'model_outputs.txt'")

📝 Outputs saved to 'model_outputs.txt'


# Step 6: Download the bechmarks

In [ ]:
import subprocess, time, socket, re, os, urllib.request
from pathlib import Path
from IPython.display import display, HTML

PORT        = 8989
SERVE_DIR   = "/kaggle/working"
SEARCH_ROOT = Path(SERVE_DIR)
CF_BIN      = "/kaggle/working/cloudflared"

# -- Check internet -------------------------------------------------------
def internet_ok():
    try:
        socket.setdefaulttimeout(5)
        socket.create_connection(("8.8.8.8", 53))
        return True
    except OSError:
        return False

if not internet_ok():
    print("No internet. Enable it in Kaggle Settings (Right Panel).")
    raise SystemExit

# -- Find Benchmark Files (Modified for JPG/TXT) ---------------------------
# We look for the JPG graph and the text outputs created in Part 2
files_to_download = sorted(list(SEARCH_ROOT.glob("*.jpg")) + list(SEARCH_ROOT.glob("*_output.txt")))

if not files_to_download:
    print("No benchmark files found. Run Part 2 first.")
    raise SystemExit

# -- Download cloudflared --------------------------------------------------
if not Path(CF_BIN).exists():
    print("Downloading cloudflared ...")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", CF_BIN
    ], check=True)
    os.chmod(CF_BIN, 0o755)

# -- Start HTTP server -----------------------------------------------------
print(f"Starting server on port {PORT} ...")
server_proc = subprocess.Popen(
    ["python", "-m", "http.server", str(PORT), "--directory", SERVE_DIR, "--bind", "0.0.0.0"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Verify server
server_alive = False
for _ in range(10):
    time.sleep(1)
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/", timeout=3)
        server_alive = True
        break
    except: pass

if not server_alive:
    print("HTTP server failed.")
    raise SystemExit

# -- Start Cloudflare tunnel -----------------------------------------------
print("Starting Cloudflare tunnel (Wait 15s)...")
cf_log = "/kaggle/working/cf_tunnel.log"
if Path(cf_log).exists(): Path(cf_log).unlink()

cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate", "--logfile", cf_log],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

tunnel_url = None
for _ in range(60):
    time.sleep(1)
    if Path(cf_log).exists():
        log_text = Path(cf_log).read_text(errors="ignore")
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', log_text)
        if m:
            tunnel_url = m.group(0)
            break

if not tunnel_url:
    print("Tunnel failed.")
    raise SystemExit

# -- Display HTML Links ----------------------------------------------------
rows = ""
for f in files_to_download:
    rel = f.name
    url = f"{tunnel_url}/{rel}"
    rows += (f"<tr><td><b>{rel}</b></td>"
             f"<td><a href='{url}' target='_blank'>Download/View</a></td>"
             f'<td><code>wget "{url}"</code></td></tr>')

display(HTML(f"""
<h3>Nayari Benchmark Downloads</h3>
<table border='1' cellpadding='6' cellspacing='0' style='border-collapse:collapse;font-family:sans-serif;'>
  <tr style='background:#dee2e6'><th>File Name</th><th>Action</th><th>Command</th></tr>
  {rows}
</table>
<p>Public Folder: <a href='{tunnel_url}' target='_blank'>{tunnel_url}</a></p>
"""))